In [ ]:
import pandas as pd
import statsmodels.api as sm

# 1. Load model-ready data (already cleaned + dummified + types fixed)
df = pd.read_csv("kidney_disease_model_ready.csv")

# 2. Define response variable
y = df["sc"]  # serum creatinine

# 3. Build full predictor matrix (drop id + response)
X_full = df.drop(columns=["id", "sc"])

# 4. Add intercept and fit FULL model
X_full_const = sm.add_constant(X_full)
full_model = sm.OLS(y, X_full_const).fit()

print("===== FULL MODEL SUMMARY =====")
print(full_model.summary())

# 5. Get p-values from full model (drop intercept)
pvals = full_model.pvalues.drop("const")

# 6. Automatically select predictors with p < 0.05
#    and EXCLUDE 'classification_notckd' on purpose
selected_predictors = [
    var for var, p in pvals.items()
    if (p < 0.05) and (var != "classification_notckd")
]


===== FULL MODEL SUMMARY =====
                            OLS Regression Results                            
Dep. Variable:                     sc   R-squared:                       0.876
Model:                            OLS   Adj. R-squared:                  0.853
Method:                 Least Squares   F-statistic:                     39.10
Date:                Tue, 25 Nov 2025   Prob (F-statistic):           9.47e-49
Time:                        16:59:04   Log-Likelihood:                -236.48
No. Observations:                 158   AIC:                             523.0
Df Residuals:                     133   BIC:                             599.5
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------

In [2]:
print("\n===== SELECTED PREDICTORS FOR REDUCED MODEL (p < 0.05, excluding classification) =====")
print(selected_predictors)

# 7. Build reduced X with only selected predictors
X_reduced = df[selected_predictors]
X_reduced_const = sm.add_constant(X_reduced)

# 8. Fit REDUCED model
reduced_model = sm.OLS(y, X_reduced_const).fit()

print("\n===== REDUCED MODEL SUMMARY =====")
print(reduced_model.summary())



===== SELECTED PREDICTORS FOR REDUCED MODEL (p < 0.05, excluding classification) =====
['al', 'bu', 'pot', 'wc', 'pc_normal', 'pe_yes', 'ane_yes']

===== REDUCED MODEL SUMMARY =====
                            OLS Regression Results                            
Dep. Variable:                     sc   R-squared:                       0.854
Model:                            OLS   Adj. R-squared:                  0.847
Method:                 Least Squares   F-statistic:                     125.0
Date:                Tue, 25 Nov 2025   Prob (F-statistic):           2.66e-59
Time:                        16:59:38   Log-Likelihood:                -249.49
No. Observations:                 158   AIC:                             515.0
Df Residuals:                     150   BIC:                             539.5
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef    st

In [3]:
anova_results = sm.stats.anova_lm(reduced_model, full_model)
print(anova_results)

   df_resid         ssr  df_diff    ss_diff         F    Pr(>F)
0     150.0  217.626704      0.0        NaN       NaN       NaN
1     133.0  184.579206     17.0  33.047498  1.400743  0.145709
